# Task 1 — Fine-tune Qwen2.5-Coder-7B-Instruct

Notebook này chạy **QLoRA 4-bit** trên đúng 26 bài train và đánh giá trên 6 bài validation. Nó pull branch `task-1`, tải ZIP từ [thư mục Drive của cuộc thi](https://drive.google.com/drive/folders/1kBd4pE4JHcDtm0WFYueFq6aTEuTanUNC), lấy `sample_dataset.zip` bên trong và chuẩn bị dữ liệu trên Drive cá nhân. Chọn runtime **GPU** trong Colab. Dữ liệu và adapter không được commit lên repo.

Quy trình: cài thư viện → pull `task-1` → tải và chuẩn bị bộ 32 bài có nhãn → kiểm tra dữ liệu/token → smoke test 1 bước → train 3 epoch → xem checkpoint tốt nhất. Nếu link folder không tải được bằng `gdown`, đặt ZIP ngoài vào MyDrive và khai báo `DRIVE_ZIP` ở cell dữ liệu.

In [ ]:
%pip -q install "transformers>=4.46,<5" "peft>=0.13,<1" "accelerate>=0.34,<2" "bitsandbytes>=0.45,<1" "pyyaml>=6" "pydantic>=2.5" gdown

In [ ]:
from google.colab import drive
from pathlib import Path
import json, subprocess, sys
import torch

drive.mount('/content/drive')
assert torch.cuda.is_available(), 'Chọn Runtime > Change runtime type > GPU rồi chạy lại.'
print('GPU:', torch.cuda.get_device_name(0))

PROJECT_HOME = Path('/content/drive/MyDrive/LLM/Final Project')
REPO_URL = 'https://github.com/tiendinhquang2104/llm-final-project.git'
REPO_BRANCH = 'task-1'
REPO_DIR = Path('/content/llm-final-project-task1-finetune')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_BRANCH, '--single-branch', REPO_URL, str(REPO_DIR)], check=True)
else:
    if not (REPO_DIR / '.git').is_dir():
        raise RuntimeError(f'{REPO_DIR} đã tồn tại nhưng không phải git repo')
    origin = subprocess.check_output(['git', '-C', str(REPO_DIR), 'remote', 'get-url', 'origin'], text=True).strip()
    if origin != REPO_URL:
        raise RuntimeError(f'Git origin không đúng: {origin}')
    if subprocess.check_output(['git', '-C', str(REPO_DIR), 'status', '--porcelain'], text=True).strip():
        raise RuntimeError('Checkout Colab có thay đổi chưa lưu; xử lý trước khi pull.')
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', REPO_BRANCH], check=True)
    branches = subprocess.check_output(['git', '-C', str(REPO_DIR), 'branch', '--list', REPO_BRANCH], text=True).strip()
    if branches:
        subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', REPO_BRANCH], check=True)
    else:
        subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', '-b', REPO_BRANCH, '--track', f'origin/{REPO_BRANCH}'], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', REPO_BRANCH], check=True)
required_source = ['scripts/train_task1.py', 'src/task1/finetune.py', 'configs/task1/lora_7b.yaml', 'scripts/prepare_data.py']
missing_source = [name for name in required_source if not (REPO_DIR / name).is_file()]
if missing_source:
    raise FileNotFoundError(f'Branch task-1 trên GitHub chưa có code fine-tune: {missing_source}. Push branch rồi chạy lại.')
commit = subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', '--short', 'HEAD'], text=True).strip()
sys.path = [str(REPO_DIR)] + [p for p in sys.path if p != str(REPO_DIR)]
for module in list(sys.modules):
    if module == 'src' or module.startswith('src.'):
        del sys.modules[module]
print('Source:', REPO_DIR, '| branch:', REPO_BRANCH, '| commit:', commit)

In [ ]:
import gdown, shutil, zipfile
from src.data.loader import resolve_dataset_root

FOLDER_URL = 'https://drive.google.com/drive/folders/1kBd4pE4JHcDtm0WFYueFq6aTEuTanUNC'
ZIP_NAME = 'public-test-cham-diem-phan-hoi-files.zip'
DRIVE_ZIP = None  # Nếu gdown không tải được: Path('/content/drive/MyDrive/.../public-test-cham-diem-phan-hoi-files.zip')
DOWNLOAD_DIR = Path('/content/task1-finetune-download')
EXTRACT_DIR = Path('/content/task1-finetune-sample')
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
archive = Path(DRIVE_ZIP) if DRIVE_ZIP else next(iter(DOWNLOAD_DIR.rglob(ZIP_NAME)), None)
if archive is None:
    gdown.download_folder(url=FOLDER_URL, output=str(DOWNLOAD_DIR), quiet=False)
    archive = next(iter(DOWNLOAD_DIR.rglob(ZIP_NAME)), None)
if archive is None or not zipfile.is_zipfile(archive):
    raise RuntimeError('Không thấy ZIP hợp lệ. Kiểm tra quyền xem folder hoặc đặt DRIVE_ZIP tới bản sao trong MyDrive.')
with zipfile.ZipFile(archive) as outer:
    candidates = [name for name in outer.namelist() if Path(name).name == 'sample_dataset.zip']
    if len(candidates) != 1:
        raise FileNotFoundError(f'ZIP ngoài phải có đúng một sample_dataset.zip; tìm thấy {candidates}')
    inner_zip = DOWNLOAD_DIR / 'sample_dataset.zip'
    with outer.open(candidates[0]) as source, inner_zip.open('wb') as target:
        shutil.copyfileobj(source, target)
if not zipfile.is_zipfile(inner_zip):
    raise RuntimeError('sample_dataset.zip bên trong bị lỗi')
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(inner_zip) as zf:
    root = EXTRACT_DIR.resolve()
    if any(not (root / name).resolve().is_relative_to(root) for name in zf.namelist()):
        raise ValueError('sample_dataset.zip chứa đường dẫn không an toàn')
    zf.extractall(EXTRACT_DIR)
SAMPLE_ROOT = resolve_dataset_root(EXTRACT_DIR)
DATA_HOME = PROJECT_HOME / 'data/task1_finetune_sample'
PROCESSED_DIR = DATA_HOME / 'processed'
SPLITS_DIR = DATA_HOME / 'splits'
OUTPUT_DIR = PROJECT_HOME / 'data/outputs/task1/qwen25_coder7b_qlora'
command = [sys.executable, str(REPO_DIR / 'scripts/prepare_data.py'), '--data_dir', str(SAMPLE_ROOT),
           '--output_dir', str(PROCESSED_DIR), '--splits_dir', str(SPLITS_DIR), '--seed', '42', '--val_ratio', '0.2']
subprocess.run(command, cwd=REPO_DIR, check=True)
manifest = json.loads((PROCESSED_DIR / 'manifest.json').read_text(encoding='utf-8'))
assert (manifest['n_samples'], manifest['n_train_ids'], manifest['n_val_ids']) == (32, 26, 6), manifest
print('Training source:', archive, '→', SAMPLE_ROOT)
print('Task 1 train/val:', PROCESSED_DIR / 'task1_train.jsonl', PROCESSED_DIR / 'task1_val.jsonl')
print('Output:', OUTPUT_DIR)

In [ ]:
RUNNER = REPO_DIR / 'scripts/train_task1.py'
CONFIG = REPO_DIR / 'configs/task1/lora_7b.yaml'

def run_task1(*flags):
    command = [sys.executable, str(RUNNER), '--config', str(CONFIG),
               '--processed-dir', str(PROCESSED_DIR), '--splits-dir', str(SPLITS_DIR),
               '--output-dir', str(OUTPUT_DIR), *flags]
    subprocess.run(command, cwd=REPO_DIR, check=True)

run_task1('--preflight')

## Smoke test và huấn luyện đầy đủ

Smoke test dùng một optimizer step và một bài validation, lưu trong thư mục `smoke/`. Lượt đầy đủ dùng cả 26/6 bài, lưu checkpoint mỗi epoch. Nếu Colab mất kết nối giữa chừng, kiểm tra thư mục output trước khi chạy lại để tránh ghi đè kết quả cần giữ.

In [ ]:
RUN_SMOKE = True
if RUN_SMOKE:
    run_task1('--smoke')
    smoke_summary = json.loads((OUTPUT_DIR / 'smoke/training_summary.json').read_text(encoding='utf-8'))
    print('Smoke test OK. QWK:', smoke_summary['selected_metrics']['qwk'])

In [ ]:
RUN_FULL = True
if RUN_FULL:
    run_task1()
    summary = json.loads((OUTPUT_DIR / 'training_summary.json').read_text(encoding='utf-8'))
    print('Best epoch:', summary['selected_epoch'])
    print('Validation QWK:', summary['selected_metrics']['qwk'])
    print('Validation MAE:', summary['selected_metrics']['mae'])
    print('Valid output rate:', summary['selected_metrics']['valid_output_rate'])
    print('Adapter:', OUTPUT_DIR / 'best_adapter')
    print('All metrics:', OUTPUT_DIR / 'training_summary.json')